# Tobac analysis of upward SW flux (sod_t - sob_t)

## Notes on Datastructure 

- **field**:
input datafield as an xarray.DataArray (eg. sodt_t or tqc_dia)
- **Features**:
Result of feature detection ( via feature_detection_multithreshold): point features per timestep (DataFrame). Includes time/frame, position (lat, lon), threshold_value. No cell areas yet—just points.
- **Mask_field**:
Result of segmentation (via segmentation_2D): label mask as an xarray.DataArray with the same space/time dims. Per time: 0 = background, >0 = object ID (cell). (can derive cell sizes by counting pixels per ID)
- **Features_field**:
Second return from segmentation_2D: an enriched Features DataFrame that links point features to their object info (e.g., cell ID and ncells = number of pixels in the cell). Use this to get sizes directly (area = ncells × pixel_area)


In [ ]:
# system libs
import os, sys, glob
import datetime

import tobac
import xarray as xr    
import numpy as np
if not hasattr(np, "int"):   np.int = int   # use this for the np.int problem

import matplotlib.pyplot as plt

print("numpy:", np.__version__) #np.int which was removed in nuumpy version 2.x.x which is needed for convert_timevec tool (solution: downgraded np-version to <2 in the tobac env)
print("tobac:", tobac.__version__)

print("Python:", sys.executable)
assert "conda/envs/tobac" in sys.executable, "Not using the tobac kernel — switch to 'tobac (Levante)'."


# array operators and netcdf datasets
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

# to have tools to format time
sys.path.append( '/work/bb1224/2024_MS-COURSE/tools/analysis' )
from tools import convert_timevec

## Open Datasets

In [ ]:
# Path
#sim_path = '/work/bb1376/user/daniel/icon-build/experiments/cesar1-20240806-exp010'
sim_path = '/work/bb1376/user/daniel/icon-build/experiments/cesar2-20240928-exp010'
#sim_path = '/work/bb1376/user/daniel/icon-build/experiments/cesar3-20241110-exp010'


In [ ]:
#radiation varibles.
ds_2dicon1rad = xr.open_mfdataset(f'{sim_path}/2d_rad_DOM01_ML_????????T????00Z_regrid0005d.nc')
ds_2dicon2rad = xr.open_mfdataset(f'{sim_path}/2d_rad_DOM02_ML_????????T????00Z_regrid0005d.nc')
ds_2dicon3rad = xr.open_mfdataset(f'{sim_path}/2d_rad_DOM03_ML_????????T????00Z_regrid0005d.nc')

ds_2dicon1cloud = xr.open_mfdataset(f'{sim_path}/2d_cloud_DOM01_ML_????????T????00Z_regrid0005d.nc')
ds_2dicon2cloud = xr.open_mfdataset(f'{sim_path}/2d_cloud_DOM02_ML_????????T????00Z_regrid0005d.nc')
ds_2dicon3cloud = xr.open_mfdataset(f'{sim_path}/2d_cloud_DOM03_ML_????????T????00Z_regrid0005d.nc')

In [ ]:
# Define dictionaries to store datasets

datasets2drad = {
    "exp010 icon d1 rad": ds_2dicon1rad,
    "exp010 icon d2 rad": ds_2dicon2rad,
    "exp010 icon d3 rad": ds_2dicon3rad,
}

datasets2dcloud = {
    "exp010 icon d1 rad": ds_2dicon1cloud,
    "exp010 icon d2 rad": ds_2dicon2cloud,
    "exp010 icon d3 rad": ds_2dicon3cloud,
}


In [ ]:
# Define Lindenberg coordinates:
lon_lind, lat_lind = 14.11845, 52.20967

# Define the spatial range over Lindenberg:
lat_min, lat_max = 51.85, 52.55    
lon_min, lon_max = 13.65, 14.55 

# Apply spatial selection to all datasets
for datasets in [datasets2drad]:
    for name, ds in datasets.items():
        datasets[name] = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))


In [ ]:
# Format time 
for name, ds in datasets2drad.items():
    ds['time'] = convert_timevec(ds.time.data)

In [ ]:
ds_2dicon1rad

## upward SW flux (sod_t - sob_t)

In [ ]:
# pick one domain to start 
ds_dom1 = datasets2drad["exp010 icon d1 rad"]
ds_dom2 = datasets2drad["exp010 icon d2 rad"]
ds_dom3 = datasets2drad["exp010 icon d3 rad"]

#define field for upward shortwave flux at TOA 
field_dom1 = ds_dom1["sod_t"] - ds_dom1["sob_t"]
field_dom2 = ds_dom2["sod_t"] - ds_dom2["sob_t"]
field_dom3 = ds_dom3["sod_t"] - ds_dom3["sob_t"]

In [ ]:
field_dom1

### plot Map and Histogram

In [ ]:
field = field_dom1

# time indices
idxs = [ 10, 12, 14, 16,18,20,22,24]
idxs = [i for i in idxs if i < field.sizes["time"]]

#shared color limits across all rows (simple: min/max over those frames)
sub = field.isel(time=idxs)
vmin = float(sub.min())
vmax = float(sub.max())

# plot
fig, axs = plt.subplots(ncols=2, nrows=len(idxs), figsize=(10, 4*len(idxs)))
plt.subplots_adjust(hspace=0.5)

for i, itime in enumerate(idxs):
    # left: map with shared clim
    im = field.isel(time=itime).plot(ax=axs[i, 0], vmin=vmin, vmax=vmax)
    im.colorbar.set_label("upward shortwave flux at TOA [W m-2]")  # rename cbar

    # right: histogram (same x-range for consistency)
    field.isel(time=itime).plot.hist(ax=axs[i, 1], bins=60, range=(vmin, vmax))
    axs[i, 1].set_xlabel("upward shortwave flux at TOA [W m-2]")  # rename xlabel
    axs[i, 1].set_yscale("log")

plt.tight_layout()



## get spacings manually (dxy, dt)
tobac.utils.get_spacings() needs x-y data not lat-lon

In [ ]:
#gets spacings manually

field_cp = field.copy()

# mean latitude to convert lon->meters
lat0 = float(field_cp.lat.mean())
print(lat0)

# median step sizes in degrees 
dlat_deg = float(abs(field_cp.lat.diff("lat").median()))
dlon_deg = float(abs(field_cp.lon.diff("lon").median()))

# degrees -> meters (Earth-mean approximations)
m_per_deg_lat = 110_574.0
m_per_deg_lon = 111_320.0 * np.cos(np.deg2rad(lat0))

dy = dlat_deg * m_per_deg_lat
dx = dlon_deg * m_per_deg_lon


# one representative horizontal spacing (meters)
dxy = float(np.sqrt(dx * dy))   # geometric mean (area-preserving)
print(f"dxy ≈ {dxy:.1f} m (dx={dx:.1f}, dy={dy:.1f})")

#timesteps (seconds)
dt = float((field_cp.time.diff('time') / np.timedelta64(1, 's')).median().item())
print("dt [s]:", dt)

## Feature detection (multiple treshhold)

In [ ]:
# Dictionary containing keyword arguments for feature detection step (Keywords could also be given directly in the function call).
parameters_features={}
parameters_features['position_threshold']='weighted_diff'
parameters_features['sigma_threshold']=0.5
parameters_features['n_min_threshold']=4
#parameters_features['target']='minimum'
parameters_features['target']='maximum'
parameters_features['threshold']=[400,350,300,250,200]


In [ ]:
# Perform feature detection:
print('starting feature detection')
Features = tobac.feature_detection_multithreshold(field, dxy, **parameters_features)
#Features.to_hdf(savedir / 'Features.h5', 'table')
print('feature detection performed')

In [ ]:
Features


In [ ]:
#Plot
frame= 20

#field[frame].plot(cmap="binary") #black and white cmap
field[frame].plot(cbar_kwargs={"label": "upward shortwave flux at TOA [W m$^{-2}$]"})
points = {
    threshold:plt.plot(ft.lon, ft.lat, "o")[0]
    for threshold, ft in Features[Features.frame==frame].groupby("threshold_value")
}
plt.legend(list(points.values()), list(points.keys()), title="Flux [W m-2]")


## Segmentation

In [ ]:
# Dictionary containing keyword options for the segmentation step:
parameters_segmentation={}
#parameters_segmentation['target']='minimum'
parameters_segmentation['target']='maximum'
parameters_segmentation['method']='watershed'
parameters_segmentation['threshold']=300


In [ ]:
# Perform segmentation and save results:
print('Starting segmentation based on field.')
Mask_field, Features_field = tobac.segmentation_2D(Features, field, dxy, **parameters_segmentation)
print('segmentation performed')
#Mask_OLR.to_netcdf(savedir / 'Mask_Segmentation_OLR.nc', encoding={"segmentation_mask":{"zlib":True, "complevel":4}})
#Features_OLR.to_hdf(savedir / 'Features_OLR.h5', 'table')

In [ ]:
plt.figure(figsize=(8.8, 4.8))
field[frame].plot(cmap="binary", cbar_kwargs={"label": "upward shortwave flux at TOA [W m$^{-2}$]"})
Mask_field[frame].where(Mask_field[frame]>0).plot(cmap="tab20", alpha=0.75)

## Cell-size- distribution 

In [ ]:
#total distibution across all frames
sizes=Features_field["ncells"] * (dxy**2) # m² per cell

# histogram (cell-size distribution)
plt.figure(figsize=(6,4))
plt.hist(sizes/1e6, bins=40) # convert to km²
plt.xlabel("Cell Size [km$^2$]"); plt.ylabel("Count")
#plt.yscale("log")  
plt.tight_layout() 
plt.show()

In [ ]:
# distribution in indivudual frame
frame = 20  
sizes = Features_field.loc[Features_field["frame"] == frame, "ncells"] * (dxy**2)  # m²

plt.figure(figsize=(6,4))
plt.hist(sizes/1e6, bins=40)  # km²
plt.xlabel("Cell Size [km$^2$]"); plt.ylabel("Count")
# plt.yscale("log")
plt.tight_layout()
plt.show()


## Domain comparison of cell size distribution


In [ ]:
# Dictionary containing keyword options for the segmentation step:
parameters_segmentation={}
#parameters_segmentation['target']='minimum'
parameters_segmentation['target']='maximum'
parameters_segmentation['method']='watershed'
parameters_segmentation['threshold']=325


In [ ]:
time= "2024-09-28 14:00:00"
domains = [("DOM01", field_dom1), ("DOM02", field_dom2), ("DOM03", field_dom3)]

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(12, 12), constrained_layout=True)
fig.suptitle(f"Upward SW flux TOA - Domain comparison — {time}", fontsize=14, fontweight="bold")
for i, (name, field) in enumerate(domains):    
    # Perform feature detection:
    Features = tobac.feature_detection_multithreshold(field, dxy, **parameters_features)

    # Perform segmentation:
    Mask_field, Features_field = tobac.segmentation_2D(Features, field, dxy, **parameters_segmentation)

    # LEFT: field (with colorbar label) + mask overlay
    #field[frame].plot(ax=axs[i,0], cmap="binary")
    #Mask_field[frame].where(Mask_field[frame]>0).plot(ax=axs[i,0], cmap="tab20", alpha=0.75)

    # plot by timestamp, not by frame , becasue frames do not align (different starting points for domains)
    field.sel(time=time).plot(ax=axs[i,0], cmap="binary", cbar_kwargs={"label": "upward shortwave flux at TOA [W m$^{-2}$]"})
    Mask_field.sel(time=time).where(Mask_field.sel(time=time)>0).plot(ax=axs[i,0], cmap="tab20", alpha=0.75)
    #axs[i,0}set_title(f
    
    # RIGHT: histogram of cell sizes
    sizes = Features_field.loc[Features_field["time"] == time, "ncells"] * (dxy**2)  # m²
    axs[i,1].hist(np.asarray(sizes)/1e6, bins=40)  # km²
    axs[i,1].set_xlabel("Cell Size [km$^2$]")
    axs[i,1].set_ylabel("Count")

    #Titles
    axs[i,0].set_title(f"{name} — {time}")
    axs[i,1].set_title(f"{name} — cell-size distribution")

plt.show()


## Timeseries of Cell-Size Distribution

In [ ]:
# pick one domain to start 
ds_dom1 = datasets2drad["exp010 icon d1 rad"]
ds_dom2 = datasets2drad["exp010 icon d2 rad"]
ds_dom3 = datasets2drad["exp010 icon d3 rad"]

#define field for upward shortwave flux at TOA devided by shortwave flux downward -> ALBEDO
field_dom1 = (ds_dom1["sod_t"] - ds_dom1["sob_t"]) / ds_dom1["sod_t"]
field_dom2 = (ds_dom2["sod_t"] - ds_dom2["sob_t"]) / ds_dom2["sod_t"]
field_dom3 = (ds_dom3["sod_t"] - ds_dom3["sob_t"]) / ds_dom3["sod_t"]

In [ ]:
field = field_dom1

# time indices
idxs = [ 10, 12, 14, 16,18,20,22,24]
idxs = [i for i in idxs if i < field.sizes["time"]]

#shared color limits across all rows (simple: min/max over those frames)
sub = field.isel(time=idxs)
vmin = float(sub.min())
vmax = float(sub.max())

# plot
fig, axs = plt.subplots(ncols=2, nrows=len(idxs), figsize=(10, 4*len(idxs)))
plt.subplots_adjust(hspace=0.5)

for i, itime in enumerate(idxs):
    # left: map with shared clim
    im = field.isel(time=itime).plot(ax=axs[i, 0], vmin=vmin, vmax=vmax)
    im.colorbar.set_label("Albedo")  # rename cbar

    # right: histogram (same x-range for consistency)
    field.isel(time=itime).plot.hist(ax=axs[i, 1], bins=60, range=(vmin, vmax))
    axs[i, 1].set_xlabel("Albedo")  # rename xlabel
    axs[i, 1].set_yscale("log")

plt.tight_layout()



In [ ]:
field_dom1.time.values



In [ ]:
# Dictionary containing keyword arguments for feature detection step (Keywords could also be given directly in the function call).
parameters_features={}
parameters_features['position_threshold']='weighted_diff'
parameters_features['sigma_threshold']=0.5
parameters_features['n_min_threshold']=4
#parameters_features['target']='minimum'
parameters_features['target']='maximum'
parameters_features['threshold']=[0.9 ,0.7, 0.5, 0.3, 0.1]

# Dictionary containing keyword options for the segmentation step:
parameters_segmentation={}
#parameters_segmentation['target']='minimum'
parameters_segmentation['target']='maximum'
parameters_segmentation['method']='watershed'
parameters_segmentation['threshold']= 0.4


In [ ]:
times = [
    '2024-09-28T14:00:00.000000000', '2024-09-28T14:05:00.000000000',
    '2024-09-28T14:10:00.000000000', '2024-09-28T14:15:00.000000000',
    '2024-09-28T14:20:00.000000000', '2024-09-28T14:25:00.000000000',
    '2024-09-28T14:30:00.000000000', '2024-09-28T14:35:00.000000000',
    '2024-09-28T14:40:00.000000000', '2024-09-28T14:45:00.000000000',
    '2024-09-28T14:50:00.000000000', '2024-09-28T14:55:00.000000000',
]

domain = field_dom1
domain_name = "DOM1"



# 1) Feature detection + segmentation for the whole field 
Features = tobac.feature_detection_multithreshold(domain, dxy, **parameters_features)

Mask_field, Features_field = tobac.segmentation_2D(Features, domain, dxy, **parameters_segmentation)


#  2) Precompute all sizes to get common histogram bins 

sizes_per_time = []

for t in times:
    # select features for this time
    sizes_t = Features_field.loc[Features_field["time"] == t, "ncells"] * (dxy**2)  # m²
    sizes_km2 = np.asarray(sizes_t) / 1e6  # km²
    sizes_per_time.append(sizes_km2)


# 3) Plot 

n_times = len(times)
fig, axs = plt.subplots(nrows=n_times, ncols=2, figsize=(12, 4 * n_times), constrained_layout=True)
fig.suptitle(f"{domain_name} — Timestamp comparrison of Albedo and Cell-Size distribution", fontsize=14, fontweight="bold")

for i, (t, sizes_km2) in enumerate(zip(times, sizes_per_time)):
    # field & mask at this time
    field_t = domain.sel(time=t)
    mask_t = Mask_field.sel(time=t)

    # LEFT: field with mask overlay
    field_t.plot(ax=axs[i, 0], cmap="binary", cbar_kwargs={"label": "Albedo"})
    mask_t.where(mask_t > 0).plot(ax=axs[i, 0], cmap="tab20", alpha=0.75)
    axs[i, 0].set_title(f"{domain_name} — {t}")

    # RIGHT: histogram of cell sizes
    axs[i, 1].hist(sizes_km2)#, bins=bins)
    axs[i, 1].set_xlabel("Cell size [km$^2$]")
    axs[i, 1].set_ylabel("Count")
    axs[i, 1].set_title(f"{domain_name} — cell-size distribution @ {t}")

plt.show()





## Cell-Size Distribution over all timestamps

In [ ]:

# ncells → physical area [m²]
sizes_all = Features_field["ncells"] * (dxy**2)

# convert to km²
sizes_all_km2 = np.asarray(sizes_all) / 1e6

print(f"Number of cells in all selected times: {sizes_all_km2.size}")

# Plot: cell-size distribution over all timestamps for DOM1 

fig, ax = plt.subplots(figsize=(7, 5))

ax.hist(sizes_all_km2, bins=bins, edgecolor="black", alpha=0.7)
ax.set_xlabel("Cell size [km$^2$]")
ax.set_ylabel("Count")
ax.set_title(f"{domain_name} — Cell-size distribution over all Timestamps")

plt.tight_layout()
plt.show()


In [ ]:

from scipy.stats import expon

# --- Convert to km² ---
sizes_all = Features_field["ncells"] * (dxy**2)
sizes_all_km2 = np.asarray(sizes_all) / 1e6

print(f"Total number of detected cells: {sizes_all_km2.size}")


# --- Fit exponential distribution ---
mean_size = sizes_all_km2.mean()
lambda_hat = 1 / mean_size   # MLE estimator

print(f"Estimated lambda = {lambda_hat:.4f}  (mean = {mean_size:.2f} km²)")

# x-values for smooth curve
x_vals = np.linspace(0, max_size, 300)

# exponential PDF
pdf_vals = lambda_hat * np.exp(-lambda_hat * x_vals)

# Scale PDF to match histogram counts (multiply by total number * bin width)
bin_width = bins[1] - bins[0]
pdf_hist_scaled = pdf_vals * sizes_all_km2.size * bin_width


# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 5))

# histogram
ax.hist(sizes_all_km2, bins=bins, edgecolor="black", alpha=0.7, label="Observed")

# exponential curve
ax.plot(x_vals, pdf_hist_scaled, "r-", linewidth=2,
        label=f"Exponential fit (λ={lambda_hat:.3f})")

ax.set_xlabel("Cell size [km$^2$]")
ax.set_ylabel("Count")
ax.set_title(f"{domain_name} — Cell-size distribution with exponential fit")
ax.legend()

# --- Set x-axis limit to 500 km² ---
ax.set_xlim(0, 500)

plt.tight_layout()
plt.show()



In [ ]:
Features

## Pair Correlation Function

In [ ]:

def pair_correlation(x_points, y_points, domain_bounds, dr, r_max):
    """
    Compute the pair-correlation function g(r) for a set of points within a rectangular domain.
    - x_points, y_points: arrays of point coordinates (e.g. cloud centroids).
    - domain_bounds: tuple (x_min, x_max, y_min, y_max) defining the domain.
    - dr: width of the radial distance bins.
    - r_max: maximum distance to consider for pair counting.
    Returns: radii (bin centers) and g(r) values as numpy arrays.
    """
    x_min, x_max, y_min, y_max = domain_bounds
    Lx = x_max - x_min
    Ly = y_max - y_min
    area = Lx * Ly
    N = len(x_points)

    # Overall number density of points
    density = N / area
    
    # Set up distance bins
    edges = np.arange(0, r_max + dr, dr)
    radii = 0.5 * (edges[:-1] + edges[1:])  # bin centers for plotting
    
    # Count pair distances
    pair_counts = np.zeros(len(radii))
    for i in range(N):
        # Compute distances from point i to all other points j > i
        dx = x_points[i+1:] - x_points[i]
        dy = y_points[i+1:] - y_points[i]
        distances = np.sqrt(dx**2 + dy**2)
        # Bin the distances
        counts, _ = np.histogram(distances, bins=edges)
        pair_counts += counts  # accumulate counts for each distance bin
    
    # Expected pair counts for a random (Poisson) distribution in each bin
    # (Using homogeneous assumption: expected count = density * N * area_of_ring / 2)
    # area_of_ring between r_inner and r_outer = π(r_outer^2 - r_inner^2)
    ring_areas = np.pi * (edges[1:]**2 - edges[:-1]**2)
    expected_counts = density * N * ring_areas / 2.0
    
    # Compute g(r) as the ratio of observed to expected pair counts
    g_r = pair_counts / expected_counts
    return radii, g_r


In [ ]:
# Assuming your features table is called `features`
lat = Features['lat'].values
lon = Features['lon'].values


# Convert lat/lon to numpy arrays
lat = np.asarray(lat)
lon = np.asarray(lon)

# Earth radius
R = 6371e3  # meters

# Reference point = domain center
lat0 = np.deg2rad(np.mean(lat))
lon0 = np.deg2rad(np.mean(lon))

lat_rad = np.deg2rad(lat)
lon_rad = np.deg2rad(lon)

# Tangent-plane projection
x_km = (lon_rad - lon0) * np.cos(lat0) * R / 1000.0
y_km = (lat_rad - lat0) * R / 1000.0

x_min, x_max = x_km.min(), x_km.max()
y_min, y_max = y_km.min(), y_km.max()

domain_bounds = (x_min, x_max, y_min, y_max)




In [ ]:
radii, g_cloud = pair_correlation(
    x_km, y_km,
    domain_bounds,
    dr=5.0,      # width of distance bins in km
    r_max=400.0  # maximum distance in km
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radii, g_cloud, marker='o', linestyle='-', 
         color='tab:green', label='Reference $g(r)$')

plt.axhline(1.0, color='gray', linestyle='--', linewidth=1,
            label='$g(r)=1$ (random Poisson)')

plt.xlabel('Pair distance $r$ (km)', fontsize=12)
plt.ylabel('Pair correlation $g(r)$', fontsize=12)
plt.title('Actual Pair Correlation Function $g(r)$', fontsize=14)

plt.grid(True, linestyle=':', linewidth=0.7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#reference 

In [ ]:
N_obs = len(Features)


In [ ]:
# Parameters
realizations = 10  # number of Monte Carlo samples
dr = 5.0            # bin width for distances (e.g., 5 km)
r_max = 200.0        # maximum distance to evaluate PCF
N_obs = len(Features) # number of cloud objects identified by TOBAC


# Storage for accumulated g(r) from each realization
g_accumulated = []

for i in range(realizations):
    # Generate a new random Poisson sample of N_obs points
    xr = np.random.uniform(x_min, x_max, N_obs)
    yr = np.random.uniform(y_min, y_max, N_obs)
    _, g_r = pair_correlation(xr, yr, (x_min, x_max, y_min, y_max), dr, r_max)
    g_accumulated.append(g_r)

# Compute the mean reference g(r) across all realizations
g_ref_mean = np.mean(g_accumulated, axis=0)

plt.plot(g_ref_mean)
plt.xlabel('Pair Distance $r$ ', fontsize=12)
plt.ylabel('Pair Correlation $g(r)$', fontsize=12)
plt.title('Reference Pair Correlation Function $g(r)$', fontsize=14)
plt.tight_layout()
plt.show()